In [ ]:
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import spacy
# python -m spacy download en_core_web_sm

# Load models
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
nli_model = pipeline("text-classification", model="roberta-large-mnli")
nlp = spacy.load("en_core_web_sm")

# Simple skill extractor using SpaCy
def extract_skills(text, keywords):
    doc = nlp(text.lower())
    found = set()
    for token in doc:
        if token.text in keywords:
            found.add(token.text)
    return found

# Match score calculator
def get_match_score(candidate_text, job_text, job_skills):
    # 1. Skill overlap
    cand_skills = extract_skills(candidate_text, job_skills)
    print("Candidate Skills:",cand_skills)
    overlap = cand_skills.intersection(job_skills)
    skill_score = len(overlap) / len(job_skills) if job_skills else 0

    # 2. Semantic similarity
    embeddings = sbert_model.encode([candidate_text, job_text], convert_to_tensor=True)
    sim_score = float(util.pytorch_cos_sim(embeddings[0], embeddings[1]))

    # 3. NLI relationship
    relation = nli_model(f"{candidate_text} </s></s> {job_text}")[0]
    
    nli_bonus = 1.0 if relation['label'] == 'ENTAILMENT' else 0.5 if relation['label'] == 'NEUTRAL' else 0

    # Final weighted score (out of 100)
    
    final_score = (skill_score*4 * 40) + (sim_score * 40) + (nli_bonus * 20)
    return round(final_score, 2), {
        "skills_matched": list(overlap),
        "semantic_score": round(sim_score, 2),
        "nli_relation": relation['label'],
        'Skill Score': skill_score,
        'Sim Score': sim_score,
        'nli bonus': nli_bonus
    }

c:\Users\smitt\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Some weights of the model checkpoint at roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


Sample 1

In [ ]:
#candidate_text = "I am a data scientist with 3 years of experience in Python, SQL, and machine learning. I deployed models using AWS."
#candidate_text = "As a Software Engineer at GEP Worldwide, my efforts are focused on employing Agile and SCRUM methodologies to streamline project workflows and enhance software quality. With a solid educational foundation from the University of Mumbai, where I earned my BE in Computer Engineering, I apply rigorous technical skills in C# to develop, test, and maintain robust software solutions. My commitment to professional growth is matched by a dedication to collaborative success. The ability to troubleshoot and resolve complex issues is a cornerstone of my approach, ensuring that our team consistently delivers high-performance software that meets the dynamic needs of our clients and contributes to the organization's objectives."
#candidate_text = "While working for GEP Solution, I had an extensive work experience of 3 years in software development, module design and managing deployment cycles. I have a keen interest in exploring newer technologies and I'm proficient at C#, .NET core APIs, Angular, JavaScript and database management systems like MySQL, MongoDB, Neo4j Graph DB (NoSQL)."
#candidate_text = "I'm a tech-driven professional with 3 years of hands-on experience in software development and system design at GEP Solutions, where I led full-stack module development and deployment cycles. Transitioning from a strong foundation in software engineering, I’m now specializing in Data Science and Machine Learning, driven by a passion for turning raw data into impactful insights and intelligent solutions.My technical toolkit includes Python, C#, .NET Core APIs, SQL/MySQL, MongoDB, Neo4j, and JavaScript/Angular, along with a growing expertise in data modeling, statistical analysis, and ML algorithms. I'm currently pursuing a Master’s in Data Science, further sharpening my skills in data visualization, predictive modeling, and AI/ML deployment.I’m actively seeking opportunities where I can combine my software background with data-driven problem solving to build scalable, smart solutions. Whether it’s analyzing business trends, automating decisions, or predicting outcomes — I bring both engineering rigor and analytical thinking to the table"
candidate_text = """
I am a graphic designer with no experience in programming, machine learning, or data science. 
I do not know Python, SQL, or AWS and have never worked on model deployment or any software engineering tasks.
My work focuses only on UI design and digital illustration.
"""
#candidate_text = "With 4 years of professional experience in Data Engineering and Analytics, I decided to pursue an MS in Data Science from Northeastern University (Khoury College of Computer Sciences), Boston. My strong foundations in Computer Science and a passion for Data Engineering, Finance, and Analytics enabled me to work for JPMorgan Chase, AlphaGrep, and Quantiphi.I am now looking for my next professional challenge in Data Engineering, Data Science, and Analytics where my skills in Python, SQL, Data Engineering, Visualization, Machine Learning, and Problem Solving will be utilized for impactful projects."
#candidate_text = "Currently pursuing a Master of Science in Computer Science at Stevens Institute of Technology, I bring a strong foundation in data analysis, problem-solving, and process improvement. My current position as a Campus Recreation Assistant has further developed my organizational and time-management abilities while contributing to a collaborative campus environment.Previously, as a Consultant and Senior Analyst at Deloitte, I led business process reviews, engaged directly with clients, and delivered actionable insights that supported operational decisions. These experiences have strengthened my ability to interpret data, streamline workflows, and communicate effectively across teams.I’m always eager to learn and contribute, and I’m open to working on any task or project where I can add value. With a mindset rooted in adaptability, curiosity, and continuous improvement, I aim to bridge technical skills with business goals to drive meaningful impact."
job_text = "Looking for a machine learning engineer with experience in Python, SQL, AWS, and model deployment."

job_req_skills = {"python", "sql", "aws", "machine learning", "model deployment", "pandas", "scikit-learn"}

Emp_descrip_score, details = get_match_score(candidate_text, job_text, job_req_skills)

print("Match Score:", Emp_descrip_score)
print("Details:", details)

print('Skill_score: ',skill_score)

Candidate Skills: {'aws', 'python', 'sql'}
Match Score: 97.57
Details: {'skills_matched': ['aws', 'python', 'sql'], 'semantic_score': 0.72, 'nli_relation': 'CONTRADICTION'}
